# Output 5-1 / 5-2 — Moderate & market financing

Edit CI Summary, Input 1 market/EMBI, Input 6/7, and Macro/Ext in Excel,
save, then reload.

- **5-1** needs Chart Data + mechanical external rating (stress suite).
- **5-2** needs Input 1 flags + near-term public GFN (no stress).

See `docs/05-risk-rating.qmd`.


In [ ]:
from __future__ import annotations

from pathlib import Path

from lic_dsf.load import load_core, load_rating, load_stress
from lic_dsf.rating import (
    ChartDataRegistry,
    MarketFinancingInputs,
    assess_market_financing,
    compute_mechanical_ratings,
    market_panel,
    moderate_panel,
)
from lic_dsf.stress import run_standard_external_stress, run_standard_public_stress

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "demo":
    REPO_ROOT = REPO_ROOT.parent
WORKBOOK = REPO_ROOT / "data" / "lic-dsf-template-2025-08-12.xlsx"
WORKBOOK


In [ ]:
# Quick path: compute both Output 5 panels.
from lic_dsf.run import SHEET_5_1, SHEET_5_2, compute_outputs

output_5 = compute_outputs(WORKBOOK, include=[SHEET_5_1, SHEET_5_2])
output_5.sheets[SHEET_5_1], output_5.sheets[SHEET_5_2]

In [ ]:
macro, external, ext_base, pub_base = load_core(WORKBOOK)
stress = load_stress(WORKBOOK)
rating = load_rating(WORKBOOK)
ci = rating.ci

first = macro.inputs.first_projection_year
proj_years = list(range(first, first + 11))
ci.country, ci.dcc, round(ci.ci_score, 4), proj_years[0], proj_years[-1]


## Chart Data + mechanical rating (for 5-1)


In [ ]:
external_stress = run_standard_external_stress(
    macro, external, stress.input6, stress.residual
)
public_stress = run_standard_public_stress(
    macro, external, stress.input6, stress.residual
)

registry = ChartDataRegistry()
_EXTERNAL = (
    ("pv_debt_to_gdp", "pv_ppg_external_to_gdp"),
    ("pv_debt_to_exports", "pv_ppg_external_to_exports"),
    ("debt_service_to_exports", "ppg_debt_service_to_exports"),
    ("debt_service_to_revenue", "ppg_debt_service_to_revenue"),
)
for indicator, method in _EXTERNAL:
    registry.register_series(
        indicator,
        "baseline",
        getattr(ext_base, method)().reindex(proj_years),
        is_baseline=True,
    )
    for sid, book in external_stress.items():
        registry.register_series(
            indicator,
            sid,
            getattr(book, method)().reindex(proj_years),
            is_shock=True,
        )

registry.register_series(
    "public_pv_debt_to_gdp",
    "baseline",
    pub_base.pv_public_debt_to_gdp().reindex(proj_years),
    is_baseline=True,
)
if "B1_GDP" in public_stress:
    registry.register_series(
        "public_pv_debt_to_gdp",
        "B1_GDP",
        public_stress["B1_GDP"].pv_public_debt_to_gdp().reindex(proj_years),
        is_shock=True,
    )

mechanical = compute_mechanical_ratings(registry, ci.thresholds, years=proj_years)
mechanical.external, mechanical.overall


## Output 5-1 — Moderate risk granularity


In [ ]:
out_5_1 = moderate_panel(
    mechanical_external=mechanical.external,
    baseline_pv_gdp=ext_base.pv_ppg_external_to_gdp(),
    threshold_pv_gdp=ci.thresholds.pv_debt_to_gdp,
    rating_years=proj_years,
)
out_5_1


## Output 5-2 — Market financing


In [ ]:
gfn = pub_base.public_gfn_to_gdp().reindex(list(range(first, first + 3)))
market = assess_market_financing(
    MarketFinancingInputs(
        market_access=rating.market_access,
        gfn_to_gdp=gfn,
        embi_spread=rating.embi_spread,
    )
)
out_5_2 = market_panel(market)
out_5_2
